# 09. 얇고 긴 스크래치 필터 전략 리포트

목표는 모든 scratch를 재분류하는 것이 아니라, **얇고 긴 스크래치 유형만 후처리로 제거**하고, **조금 굵고 진하게 난 진짜 불량 스크래치**는 유지하는 것이다.

따라서 전략은 단일 threshold가 아니라 아래 순서로 잡는다.

1. 얇고 긴 형상 후보를 먼저 찾는다.
2. 굵거나 진한 raw evidence가 있으면 hard keep으로 보호한다.
3. `group` label은 운영 로직에 직접 쓰지 않고, threshold 검증 지표로만 사용한다.
4. `진불스크래치` 오제거율을 먼저 제한하고, 그 안에서 `미세스크래치` 제거율을 최대화한다.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

def configure_korean_font():
    candidates = ["Malgun Gothic", "맑은 고딕", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR", "AppleGothic"]
    available = {font.name for font in font_manager.fontManager.ttflist}
    selected = next((name for name in candidates if name in available), None)
    if selected:
        matplotlib.rcParams["font.family"] = selected
    matplotlib.rcParams["axes.unicode_minus"] = False
    return selected

print("Korean font:", configure_korean_font())
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## 09-1. 입력 설정

`CSV_PATH`에 component feature CSV 경로를 입력한다. 필요한 최소 컬럼은 `image_path`, `group`, `area`, `bbox_width`, `bbox_height`이다.

`group` 기준은 다음처럼 사용한다.

- `진불스크래치` 포함: 보호해야 하는 진짜 불량
- `미세스크래치` 포함: 제거하고 싶은 후보
- 그 외: 가능하면 제거하지 말아야 하는 기타 class

In [ ]:
ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "Vision" / "Process" / "Scratch_Postprocess",
    Path.cwd().parent,
]
PROJECT_ROOT = next((p for p in ROOT_CANDIDATES if (p / "scratch_postprocess_utils.py").exists()), Path.cwd())
RUNS_ROOT = PROJECT_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# TODO: 실제 CSV 경로를 넣는다.
CSV_PATH = ""

MICRO_KEYWORD = "미세스크래치"
TRUE_DEFECT_KEYWORD = "진불스크래치"

# 진불스크래치가 잘못 제거되는 비율의 허용 상한. 처음에는 보수적으로 작게 둔다.
MAX_TRUE_DEFECT_FALSE_REMOVE_RATE = 0.02
MAX_NON_MICRO_FALSE_REMOVE_RATE = 0.05

# contrast proxy를 강제로 지정하고 싶으면 컬럼명을 입력한다. 비워두면 자동 선택한다.
CONTRAST_PROXY = ""

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)


In [ ]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    errors = []
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            errors.append((encoding, str(exc)))
    raise RuntimeError(f"CSV를 읽지 못했습니다: {errors}")

def csv_has_required_columns(path: Path) -> bool:
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            cols = pd.read_csv(path, encoding=encoding, nrows=0).columns
            return {"image_path", "group"}.issubset(set(cols))
        except Exception:
            continue
    return False

if not CSV_PATH:
    candidates = [p for p in RUNS_ROOT.glob("*.csv") if csv_has_required_columns(p)]
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("CSV_PATH에 실제 feature CSV 경로를 입력해주세요.")
    CSV_PATH = str(candidates[-1])

CSV_PATH = Path(CSV_PATH)
raw_df = read_csv_flexible(CSV_PATH)
print("CSV_PATH:", CSV_PATH)
print("rows:", len(raw_df), "columns:", len(raw_df.columns))
display(raw_df.head())


## 09-2. Feature Engineering

이 문제에서는 contrast보다 먼저 **형상 proxy**가 중요하다.

- `bbox_minor`: 두께 proxy. 작을수록 얇다.
- `bbox_major`: 길이 proxy. 클수록 길다.
- `bbox_aspect_engineered`: `major / minor`. 클수록 길쭉하다.
- `bbox_fill_ratio_engineered`: bbox 안에서 mask가 차지하는 비율. 낮을수록 얇은 선형 구조에 가깝다.
- `contrast_proxy`: 진하게 난 스크래치를 보호하기 위한 raw evidence proxy.

In [ ]:
CAMERA_PATTERN = re.compile(r"\[([^\[\]]+)\](?=\.[^.]+$)")

def parse_camera_mode(image_path):
    name = Path(str(image_path)).name
    m = CAMERA_PATTERN.search(name)
    if m:
        return str(m.group(1))
    parts = re.findall(r"\[([^\[\]]+)\]", name)
    return str(parts[-1]) if parts else "unknown"

def first_existing(columns, candidates):
    for col in candidates:
        if col in columns:
            return col
    return None

def classify_label(group):
    text = str(group)
    if TRUE_DEFECT_KEYWORD in text:
        return "진불스크래치"
    if MICRO_KEYWORD in text:
        return "미세스크래치"
    return "기타"

def prepare_frame(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    required = ["image_path", "group"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    out = df.copy()
    out["row_id"] = np.arange(len(out))
    out["camera_mode"] = out["image_path"].map(parse_camera_mode)
    out["label_type"] = out["group"].map(classify_label)
    out["is_micro"] = out["group"].astype(str).str.contains(MICRO_KEYWORD, na=False)
    out["is_true_defect"] = out["group"].astype(str).str.contains(TRUE_DEFECT_KEYWORD, na=False)

    area_col = first_existing(out.columns, ["area", "area_px"])
    width_col = first_existing(out.columns, ["bbox_width", "bbox_w"])
    height_col = first_existing(out.columns, ["bbox_height", "bbox_h"])
    if area_col is None or width_col is None or height_col is None:
        raise ValueError("area/bbox_width/bbox_height 계열 컬럼이 필요합니다.")

    out["area_proxy"] = pd.to_numeric(out[area_col], errors="coerce").clip(lower=0)
    bw = pd.to_numeric(out[width_col], errors="coerce").clip(lower=0)
    bh = pd.to_numeric(out[height_col], errors="coerce").clip(lower=0)
    out["bbox_major"] = np.maximum(bw, bh)
    out["bbox_minor"] = np.minimum(bw, bh)
    out["bbox_aspect_engineered"] = out["bbox_major"] / (out["bbox_minor"] + 1e-6)
    out["bbox_area_engineered"] = bw * bh
    out["bbox_fill_ratio_engineered"] = out["area_proxy"] / (out["bbox_area_engineered"] + 1e-6)
    out["line_width_by_area"] = out["area_proxy"] / (out["bbox_major"] + 1e-6)
    out["log_area"] = np.log1p(out["area_proxy"])

    contrast_priority = [
        "rgb_contrast_z",
        "luma_contrast_z",
        "perimeter_skeleton_diff_euclidean",
        "perimeter_center_diff_euclidean",
        "rgb_euclidean_contrast",
        "max_channel_contrast_abs",
        "mean_channel_contrast_abs",
        "luma_contrast_abs",
    ]
    contrast_col = CONTRAST_PROXY if CONTRAST_PROXY in out.columns else first_existing(out.columns, contrast_priority)
    if contrast_col is None:
        raise ValueError("contrast proxy로 사용할 수 있는 컬럼이 없습니다.")
    out["contrast_proxy"] = pd.to_numeric(out[contrast_col], errors="coerce")
    out["log_contrast_proxy"] = np.log1p(out["contrast_proxy"].clip(lower=0))

    return out, contrast_col

df, selected_contrast_col = prepare_frame(raw_df)
print("selected contrast column:", selected_contrast_col)
display(df[["image_path", "group", "camera_mode", "label_type", "area_proxy", "bbox_major", "bbox_minor", "bbox_aspect_engineered", "bbox_fill_ratio_engineered", "contrast_proxy"]].head())


In [ ]:
summary_cols = ["area_proxy", "bbox_major", "bbox_minor", "bbox_aspect_engineered", "bbox_fill_ratio_engineered", "line_width_by_area", "contrast_proxy"]
group_summary = df.groupby("label_type")[summary_cols].agg(["count", "mean", "median", "std", "min", "max"]).round(4)
display(group_summary)
display(df.groupby(["group", "label_type"]).size().reset_index(name="count").sort_values("count", ascending=False).head(80))


## 09-3. 분포 확인

여기서는 얇고 긴 유형과 굵고 진한 유형이 실제로 어느 feature 공간에서 분리되는지 본다. 분리가 잘 되는 축은 운영 threshold 후보가 된다.

In [ ]:
LABEL_COLORS = {"진불스크래치": "#d62728", "미세스크래치": "#ff7f0e", "기타": "#4c78a8"}

def scatter_by_label(ax, x_col, y_col, title):
    for label, part in df.groupby("label_type"):
        ax.scatter(part[x_col], part[y_col], s=28, alpha=0.68, label=f"{label} ({len(part)})", color=LABEL_COLORS.get(label, "gray"))
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
scatter_by_label(axes[0, 0], "bbox_minor", "bbox_aspect_engineered", "두께 vs 길쭉함")
scatter_by_label(axes[0, 1], "bbox_minor", "contrast_proxy", "두께 vs 진함")
scatter_by_label(axes[1, 0], "bbox_aspect_engineered", "contrast_proxy", "길쭉함 vs 진함")
scatter_by_label(axes[1, 1], "line_width_by_area", "contrast_proxy", "면적 기반 폭 vs 진함")
plt.tight_layout()
plt.show()


In [ ]:
plot_cols = ["bbox_minor", "bbox_aspect_engineered", "line_width_by_area", "contrast_proxy"]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(4.3 * len(plot_cols), 4.5))
for ax, col in zip(axes, plot_cols):
    values = [pd.to_numeric(part[col], errors="coerce").dropna() for _, part in df.groupby("label_type")]
    labels = [label for label, _ in df.groupby("label_type")]
    ax.boxplot(values, labels=labels, showfliers=False)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 09-4. 운영 로직 가설

운영 rule은 다음 형태가 가장 안전하다.

```text
thin_long_candidate =
    bbox_minor <= max_thickness
    and bbox_aspect_engineered >= min_aspect
    and bbox_major >= min_length
    and bbox_fill_ratio_engineered <= max_fill_ratio

hard_keep =
    bbox_minor >= keep_if_thicker_than
    or contrast_proxy >= keep_if_stronger_than

remove = thin_long_candidate and not hard_keep
```

핵심은 `hard_keep`이다. 얇고 긴 조건을 조금 넓게 잡더라도, 굵거나 진한 component는 마지막에 다시 살린다.

In [ ]:
def clean_values(series):
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

def quantile_values(series, qs):
    values = clean_values(series)
    if values.nunique() <= 1:
        return []
    return sorted(np.unique(np.nanquantile(values, qs)).tolist())

def apply_rule(frame, rule):
    candidate = (
        (frame["bbox_minor"] <= rule["max_thickness"])
        & (frame["bbox_aspect_engineered"] >= rule["min_aspect"])
        & (frame["bbox_major"] >= rule["min_length"])
        & (frame["bbox_fill_ratio_engineered"] <= rule["max_fill_ratio"])
    )
    hard_keep = (
        (frame["bbox_minor"] >= rule["keep_if_thicker_than"])
        | (frame["contrast_proxy"] >= rule["keep_if_stronger_than"])
    )
    return candidate & ~hard_keep

def rule_metrics(frame, remove):
    remove = pd.Series(remove, index=frame.index).astype(bool)
    is_micro = frame["is_micro"].astype(bool)
    is_true = frame["is_true_defect"].astype(bool)
    non_micro = ~is_micro
    removed_count = int(remove.sum())
    return {
        "rows": int(len(frame)),
        "removed_count": removed_count,
        "removed_rate_total": float(remove.mean()) if len(remove) else np.nan,
        "micro_count": int(is_micro.sum()),
        "micro_removed_count": int((remove & is_micro).sum()),
        "micro_remove_rate": float((remove & is_micro).sum() / max(is_micro.sum(), 1)),
        "true_defect_count": int(is_true.sum()),
        "true_defect_removed_count": int((remove & is_true).sum()),
        "true_defect_false_remove_rate": float((remove & is_true).sum() / max(is_true.sum(), 1)),
        "non_micro_count": int(non_micro.sum()),
        "non_micro_removed_count": int((remove & non_micro).sum()),
        "non_micro_false_remove_rate": float((remove & non_micro).sum() / max(non_micro.sum(), 1)),
        "precision_micro_among_removed": float((remove & is_micro).sum() / max(removed_count, 1)),
    }


## 09-5. Threshold Grid Search

아래 grid는 운영 threshold 후보를 넓게 훑는다. 선택 기준은 다음 순서다.

1. `진불스크래치` 오제거율이 허용치 이하인지
2. `미세스크래치` 제거율이 높은지
3. 제거된 항목 중 미세스크래치 비율이 높은지
4. 전체 제거율이 지나치게 높지 않은지

In [ ]:
grid_values = {
    "max_thickness": quantile_values(df["bbox_minor"], np.linspace(0.05, 0.60, 8)),
    "min_aspect": quantile_values(df["bbox_aspect_engineered"], np.linspace(0.35, 0.90, 8)),
    "min_length": quantile_values(df["bbox_major"], np.linspace(0.05, 0.55, 6)),
    "max_fill_ratio": quantile_values(df["bbox_fill_ratio_engineered"], np.linspace(0.25, 0.85, 7)),
    "keep_if_thicker_than": quantile_values(df["bbox_minor"], np.linspace(0.35, 0.85, 6)),
    "keep_if_stronger_than": quantile_values(df["contrast_proxy"], np.linspace(0.50, 0.92, 7)),
}

for key, values in grid_values.items():
    if not values:
        raise ValueError(f"grid 값을 만들 수 없습니다: {key}")
    print(key, len(values), "range=", (min(values), max(values)))

rows = []
for max_thickness in grid_values["max_thickness"]:
    for min_aspect in grid_values["min_aspect"]:
        for min_length in grid_values["min_length"]:
            for max_fill_ratio in grid_values["max_fill_ratio"]:
                for keep_if_thicker_than in grid_values["keep_if_thicker_than"]:
                    for keep_if_stronger_than in grid_values["keep_if_stronger_than"]:
                        rule = {
                            "max_thickness": float(max_thickness),
                            "min_aspect": float(min_aspect),
                            "min_length": float(min_length),
                            "max_fill_ratio": float(max_fill_ratio),
                            "keep_if_thicker_than": float(keep_if_thicker_than),
                            "keep_if_stronger_than": float(keep_if_stronger_than),
                        }
                        remove = apply_rule(df, rule)
                        metrics = rule_metrics(df, remove)
                        score = (
                            metrics["micro_remove_rate"]
                            + 0.20 * metrics["precision_micro_among_removed"]
                            - 4.00 * metrics["true_defect_false_remove_rate"]
                            - 1.50 * metrics["non_micro_false_remove_rate"]
                            - 0.05 * metrics["removed_rate_total"]
                        )
                        rows.append({**rule, **metrics, "score": score})

grid_df = pd.DataFrame(rows)
grid_path = RUNS_ROOT / "thin_long_scratch_rule_grid.csv"
grid_df.to_csv(grid_path, index=False, encoding="utf-8-sig")
print("grid rows:", len(grid_df))
print("saved:", grid_path)
display(grid_df.sort_values("score", ascending=False).head(20))


In [ ]:
if df["is_true_defect"].sum() > 0:
    eligible = grid_df[grid_df["true_defect_false_remove_rate"] <= MAX_TRUE_DEFECT_FALSE_REMOVE_RATE].copy()
else:
    eligible = grid_df[grid_df["non_micro_false_remove_rate"] <= MAX_NON_MICRO_FALSE_REMOVE_RATE].copy()

if len(eligible) == 0:
    print("허용 오제거율을 만족하는 rule이 없어 score 기준으로 선택합니다. 허용치를 완화하거나 feature를 추가로 확인해야 합니다.")
    eligible = grid_df.copy()

best_rule_row = eligible.sort_values(
    ["micro_remove_rate", "precision_micro_among_removed", "score", "removed_rate_total"],
    ascending=[False, False, False, True],
).iloc[0]

rule_keys = ["max_thickness", "min_aspect", "min_length", "max_fill_ratio", "keep_if_thicker_than", "keep_if_stronger_than"]
best_rule = {key: float(best_rule_row[key]) for key in rule_keys}
best_remove = apply_rule(df, best_rule)
best_metrics = rule_metrics(df, best_remove)

display(pd.DataFrame([{**best_rule, **best_metrics}]).T.rename(columns={0: "selected_rule"}))

selected_path = RUNS_ROOT / "thin_long_selected_rule_summary.csv"
pd.DataFrame([{**best_rule, **best_metrics, "contrast_proxy_column": selected_contrast_col}]).to_csv(selected_path, index=False, encoding="utf-8-sig")
print("saved:", selected_path)


## 09-6. 선택 Rule 해석

아래 rule은 `group`을 직접 사용하지 않는다. 실제 운영에서는 component feature만 계산한 뒤 이 순서로 판단하면 된다.

In [ ]:
rule_text = f"""
### 추천 후처리 Rule

```text
thin_long_candidate = (
    bbox_minor <= {best_rule['max_thickness']:.4f}
    and bbox_aspect_engineered >= {best_rule['min_aspect']:.4f}
    and bbox_major >= {best_rule['min_length']:.4f}
    and bbox_fill_ratio_engineered <= {best_rule['max_fill_ratio']:.4f}
)

hard_keep = (
    bbox_minor >= {best_rule['keep_if_thicker_than']:.4f}
    or {selected_contrast_col} >= {best_rule['keep_if_stronger_than']:.4f}
)

remove = thin_long_candidate and not hard_keep
```

- `bbox_minor`가 작아야 제거 후보가 된다.
- `bbox_aspect_engineered`가 커야 제거 후보가 된다.
- 너무 짧은 잡음은 `bbox_major` 조건에서 제외한다.
- 굵거나 진한 경우는 `hard_keep`에서 다시 살린다.
"""
display(Markdown(rule_text))


In [ ]:
plot_df = df.copy()
plot_df["pred_remove_thin_long"] = best_remove.astype(bool)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for label, part in plot_df.groupby("label_type"):
    axes[0].scatter(part["bbox_minor"], part["bbox_aspect_engineered"], s=28, alpha=0.55, label=label, color=LABEL_COLORS.get(label, "gray"))
removed = plot_df[plot_df["pred_remove_thin_long"]]
axes[0].scatter(removed["bbox_minor"], removed["bbox_aspect_engineered"], s=80, facecolors="none", edgecolors="black", linewidths=1.4, label="remove")
axes[0].axvline(best_rule["max_thickness"], color="black", linestyle="--", linewidth=1)
axes[0].axhline(best_rule["min_aspect"], color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("bbox_minor: 두께 proxy")
axes[0].set_ylabel("bbox_aspect_engineered: 길쭉함")
axes[0].set_title("선택 rule: 형상 공간")
axes[0].legend(fontsize=8)

for label, part in plot_df.groupby("label_type"):
    axes[1].scatter(part["bbox_minor"], part["contrast_proxy"], s=28, alpha=0.55, label=label, color=LABEL_COLORS.get(label, "gray"))
axes[1].scatter(removed["bbox_minor"], removed["contrast_proxy"], s=80, facecolors="none", edgecolors="black", linewidths=1.4, label="remove")
axes[1].axvline(best_rule["keep_if_thicker_than"], color="red", linestyle="--", linewidth=1, label="thick hard keep")
axes[1].axhline(best_rule["keep_if_stronger_than"], color="red", linestyle="--", linewidth=1, label="contrast hard keep")
axes[1].set_xlabel("bbox_minor: 두께 proxy")
axes[1].set_ylabel(f"contrast_proxy: {selected_contrast_col}")
axes[1].set_title("선택 rule: hard keep 공간")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
row_pred = df.copy()
row_pred["pred_remove_thin_long"] = best_remove.astype(int)
row_pred["thin_long_candidate"] = (
    (row_pred["bbox_minor"] <= best_rule["max_thickness"])
    & (row_pred["bbox_aspect_engineered"] >= best_rule["min_aspect"])
    & (row_pred["bbox_major"] >= best_rule["min_length"])
    & (row_pred["bbox_fill_ratio_engineered"] <= best_rule["max_fill_ratio"])
).astype(int)
row_pred["hard_keep_by_thickness"] = (row_pred["bbox_minor"] >= best_rule["keep_if_thicker_than"]).astype(int)
row_pred["hard_keep_by_contrast"] = (row_pred["contrast_proxy"] >= best_rule["keep_if_stronger_than"]).astype(int)

save_cols = [
    "image_path", "component_id", "group", "camera_mode", "label_type",
    "area_proxy", "bbox_major", "bbox_minor", "bbox_aspect_engineered", "bbox_fill_ratio_engineered",
    "contrast_proxy", "thin_long_candidate", "hard_keep_by_thickness", "hard_keep_by_contrast", "pred_remove_thin_long",
]
save_cols = [c for c in save_cols if c in row_pred.columns]
row_pred_path = RUNS_ROOT / "thin_long_row_predictions.csv"
row_pred[save_cols].to_csv(row_pred_path, index=False, encoding="utf-8-sig")
print("saved:", row_pred_path)
display(row_pred[save_cols].sort_values("pred_remove_thin_long", ascending=False).head(80))


## 09-7. 카메라 촬영 방식별 영향 확인

같은 rule이라도 촬영 방식에 따라 contrast나 mask 폭이 달라질 수 있다. 아래 표에서 camera별 제거율과 진불스크래치 오제거 여부를 확인한다.

In [ ]:
camera_summary = (
    row_pred.groupby(["camera_mode", "label_type"])
    .agg(
        count=("row_id", "count"),
        removed_count=("pred_remove_thin_long", "sum"),
        removed_rate=("pred_remove_thin_long", "mean"),
        mean_thickness=("bbox_minor", "mean"),
        mean_aspect=("bbox_aspect_engineered", "mean"),
        mean_contrast=("contrast_proxy", "mean"),
    )
    .reset_index()
)
camera_summary_path = RUNS_ROOT / "thin_long_camera_group_summary.csv"
camera_summary.to_csv(camera_summary_path, index=False, encoding="utf-8-sig")
print("saved:", camera_summary_path)
display(camera_summary.sort_values(["camera_mode", "label_type"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, part in camera_summary.groupby("label_type"):
    axes[0].plot(part["camera_mode"].astype(str), part["removed_rate"], marker="o", label=label)
axes[0].set_title("카메라별 제거율")
axes[0].set_xlabel("camera_mode")
axes[0].set_ylabel("removed_rate")
axes[0].legend()

metric_by_camera = row_pred.groupby("camera_mode")[["bbox_minor", "bbox_aspect_engineered", "contrast_proxy"]].mean().reset_index()
x = np.arange(len(metric_by_camera))
axes[1].plot(metric_by_camera["camera_mode"].astype(str), metric_by_camera["bbox_minor"], marker="o", label="mean thickness")
axes[1].plot(metric_by_camera["camera_mode"].astype(str), metric_by_camera["contrast_proxy"], marker="o", label="mean contrast")
axes[1].set_title("카메라별 feature 평균")
axes[1].set_xlabel("camera_mode")
axes[1].legend()

plt.tight_layout()
plt.show()


## 결론적으로 봐야 할 것

이 문제에서는 `얇다`, `길다`, `약하다`를 동시에 만족해야 제거하고, `굵다` 또는 `진하다`가 보이면 유지하는 구조가 가장 안전하다.

검증할 때는 아래 순서로 본다.

1. `thin_long_selected_rule_summary.csv`에서 `true_defect_false_remove_rate`가 허용 가능한지 확인한다.
2. `thin_long_row_predictions.csv`에서 `진불스크래치`인데 제거된 행을 먼저 이미지로 확인한다.
3. `thin_long_camera_group_summary.csv`에서 특정 camera에서만 제거율이 튀는지 확인한다.
4. 특정 camera에서만 문제가 생기면 global threshold가 아니라 camera별 보정 threshold를 검토한다.

운영 적용은 공격적으로 제거하는 방향이 아니라, 진짜 불량 보호를 우선하는 reject-only filter로 시작하는 것이 맞다.